# Classical Model Training & Evaluation for IMU-based Exercise Classification

**Capstone -- Notebook 2 of 3.** Comparing models that classify which arm exercise is being performed from dual-IMU (forearm +
upper-arm) sensor data.

Structure: **Business Understanding -> Data Understanding -> Data Preparation -> Modeling -> Evaluation -> Findings -> Next Steps.**

## Business Understanding

**Research question:** *What is the best model for classifying which arm exercise is being performed from dual-IMU forearm +
upper-arm sensor data -- a classical model trained on aggregated per-window statistics, or a sequence model trained directly on
per-sample feature windows (notebook `3.`)?*

**This notebook's role:** train and evaluate the classical-model half of that comparison. Notebook `1.` (`1. IMU_Exercise_Windowing_and_Labeling.ipynb`)
already turned raw recordings into `data/exercise_windows.csv` -- one row per exercise-repetition window, with mean/std/min/max/rms
aggregated per feature. This notebook trains several classical classifiers on that table, tunes them with cross-validated grid
search, and evaluates the best one, so its result can be compared against notebook `3.`'s sequence model.

## Imports

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score

plt.style.use("ggplot")
%matplotlib inline

RANDOM_STATE = 42

## Data Understanding

`data/exercise_windows.csv` is notebook `1.`'s output: one row per exercise-repetition window, with `session_id` / `window_index` /
`label` / `subject` metadata plus 57 aggregated feature columns (mean/std/min/max/rms per motion feature). Re-run notebook `1.` first
if this file doesn't exist yet or is out of date with `manifest.csv`.

In [ ]:
DATA_PATH = "data/exercise_windows.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found -- run notebook 1 first")

windows_df = pd.read_csv(DATA_PATH)
print(windows_df.shape)
windows_df.head()

In [ ]:
class_counts = windows_df["label"].value_counts()
class_counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", ax=ax)
ax.set_ylabel("# windows"); ax.set_xlabel("exercise label"); ax.set_title("Windows per exercise label")
plt.tight_layout()

## Data Preparation

### Features, target, and train/test split

Meta columns (`session_id`, `window_index`, `label`, `subject`, `n_samples`, `duration_s`) are dropped from the feature matrix --
`label` is the target, and the rest aren't motion features. A held-out test set is only meaningful once there's more than one class
and enough windows per class to stratify a split; `READY_TO_MODEL` gates the modeling cells below on that, so this notebook runs
cleanly today (one class, 18 windows) and needs no code changes once more exercises are recorded.

In [ ]:
META_COLS = ["session_id", "window_index", "label", "subject", "n_samples", "duration_s"]
FEATURE_COLS = [c for c in windows_df.columns if c not in META_COLS]

X = windows_df[FEATURE_COLS]
y = windows_df["label"]

MIN_CLASS_COUNT = y.value_counts().min()
N_CLASSES = y.nunique()
READY_TO_MODEL = N_CLASSES >= 2 and MIN_CLASS_COUNT >= 2

print(f"{N_CLASSES} class(es), smallest class has {MIN_CLASS_COUNT} window(s) -> READY_TO_MODEL = {READY_TO_MODEL}")

if READY_TO_MODEL:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE,
    )
    print(f"train: {X_train.shape}, test: {X_test.shape}")
else:
    print(
        "Not enough labeled classes yet to hold out a stratified test set. "
        "See notebook 1's Findings section -- record and label at least one more exercise, "
        "rerun notebook 1, then rerun this notebook."
    )

## Modeling

Three classifiers are compared, from simplest to most flexible:

- **Dummy (most-frequent) baseline** -- the score every real model has to beat.
- **Logistic regression** (with feature standardization) -- a simple, interpretable linear baseline.
- **Random forest** and **gradient boosting** -- non-linear ensemble models that generally outperform a linear model on tabular
  motion-statistics features, at the cost of interpretability.

Each is evaluated with **stratified k-fold cross-validation** on the training set (not just a single train/test split, to reduce
variance in a comparison this small), and the random forest is additionally tuned with a **grid search** over its two most
impactful hyperparameters.

In [ ]:
if READY_TO_MODEL:
    n_splits = min(5, MIN_CLASS_COUNT)  # can't have more folds than the smallest class
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    models = {
        "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
        "logistic_regression": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]),
        "random_forest": RandomForestClassifier(random_state=RANDOM_STATE),
        "gradient_boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    }

    cv_results = {}
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro")
        cv_results[name] = scores
        print(f"{name:22s} macro-F1 = {scores.mean():.3f} +/- {scores.std():.3f}")
else:
    print("skipped -- see Data Preparation")

In [ ]:
if READY_TO_MODEL:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.boxplot(cv_results.values(), tick_labels=cv_results.keys())
    ax.set_ylabel("cross-validated macro-F1"); ax.set_title("Model comparison (cross-validation)")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
else:
    print("skipped -- see Data Preparation")

### Hyperparameter tuning: grid search on the random forest

Random forest is chosen for tuning because it's typically the strongest non-linear model on small tabular datasets like this one
and has few enough hyperparameters that a small grid search is cheap.

In [ ]:
if READY_TO_MODEL:
    param_grid = {
        "n_estimators": [100, 300],
        "max_depth": [None, 5, 10],
    }
    grid_search = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE),
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
    )
    grid_search.fit(X_train, y_train)
    print("best params:", grid_search.best_params_)
    print(f"best CV macro-F1: {grid_search.best_score_:.3f}")
    best_model = grid_search.best_estimator_
else:
    print("skipped -- see Data Preparation")

## Evaluation

**Evaluation metric: macro-averaged F1.** This is a multi-class exercise-classification problem where every class matters equally --
missing a rare exercise is just as much a failure as missing a common one -- so plain accuracy would be misleading if classes end up
imbalanced across subjects/sessions (which is likely once more data is recorded). Macro-F1 averages the per-class F1 score (the
harmonic mean of precision and recall) without weighting by class frequency, which is the right metric when every exercise class
matters regardless of how often it appears in the data.

The tuned random forest (`best_model`) is evaluated once on the held-out test set below -- the test set was never used for
cross-validation or grid search, so this is an unbiased estimate of how it'd perform on a new recording.

In [ ]:
if READY_TO_MODEL:
    y_pred = best_model.predict(X_test)
    test_f1_macro = f1_score(y_test, y_pred, average="macro")
    print(f"held-out test macro-F1: {test_f1_macro:.3f}")
    print(classification_report(y_test, y_pred))
else:
    print("skipped -- see Data Preparation")

In [ ]:
if READY_TO_MODEL:
    labels = sorted(y.unique())
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("predicted label"); ax.set_ylabel("true label"); ax.set_title("Confusion matrix -- tuned random forest")
    plt.tight_layout()
else:
    print("skipped -- see Data Preparation")

## Findings

*Template -- fill in once `READY_TO_MODEL` is `True` and the cells above have real results.*

- **Best model:** *(name the winning model and its held-out macro-F1 score in plain language, e.g. "the tuned random forest
  correctly identified the exercise X% of the time on recordings it hadn't seen during training.")*
- **Where it struggles:** *(read off the confusion matrix -- which exercise pairs get confused for each other, and a plausible
  physical reason why, e.g. two exercises with similar arm-swing patterns.)*
- **Actionable item:** *(what a non-technical reader should do with this, e.g. "this model is accurate enough to flag for human
  review, not accurate enough to auto-log reps without a spot-check.")*

## Next Steps and Recommendations

1. Once notebook `1.`'s dataset has more than one exercise class, rerun this notebook top to bottom -- `READY_TO_MODEL` will flip to
   `True` and every modeling/evaluation cell above will run for real with no code changes.
2. Compare this notebook's held-out macro-F1 against notebook `3.`'s sequence model to answer the capstone's research question.
3. If the random forest and gradient boosting scores are close, prefer gradient boosting only if its added training time is
   acceptable for how often this pipeline needs to be retrained (e.g. after every new recording session).
4. Revisit the feature set (`imu_dataset._AGG_FIELDS`) if per-class errors in the confusion matrix suggest a specific motion aspect
   (e.g. rep tempo) isn't well captured by the current mean/std/min/max/rms aggregates.